# Processus ETL — Projet Data Warehouse Films

**Source :** `merged_movies.csv`

**Destination :** Base de données PostgreSQL `movie_dwh`

**Outil ETL :** pygramETL — https://pygrametl.org/

**Tables chargées :**
- `dim_movie` — dimension film
- `dim_date` — dimension date
- `dim_studio` — dimension studio
- `dim_platform` — dimension plateforme de notation
- `fact_box_office` — faits financiers
- `fact_ratings` — faits notation

## 1. Importation des bibliothèques

In [2]:
!pip install pygrametl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import psycopg2
import pygrametl
from pygrametl.datasources import CSVSource
from pygrametl.tables import Dimension, FactTable

print('Bibliothèques importées avec succès')

Bibliothèques importées avec succès


## 2. Connexion à PostgreSQL

`psycopg2.connect()` crée une nouvelle session de base de données.
`ConnectionWrapper` encapsule la connexion pour que pygramETL puisse l'utiliser automatiquement.
`setasdefault()` définit cette connexion comme connexion par défaut pour toutes les tables pygramETL.

In [38]:
pgconn = psycopg2.connect(
    database = 'movie_DWH',
    host     = 'localhost',
    user     = 'postgres',
    password = 'system',
    port     = 5432
)

connection = pygrametl.ConnectionWrapper(pgconn)
connection.setasdefault()
connection.execute('set search_path to "Sch_DWH"')

print('Connexion établie')

Connexion établie


## 3. Extraction — Chargement du fichier source CSV

`CSVSource` lit le fichier CSV ligne par ligne.
Chaque ligne est transformée en dictionnaire Python `row` avec les colonnes comme clés.
C'est la source de données principale pour tout le pipeline ETL.

In [8]:
# Ouvrir le fichier source fusionné
fichierSource = open('merged_movies.csv', 'r', encoding='utf-8')
movies_source = CSVSource(fichierSource, delimiter=',')

print('Source CSV ouverte avec succès')

Source CSV ouverte avec succès


## 4. Transformation — Fonctions de nettoyage et formatage

 On écrit des fonctions spécifiques de transformation `f(row)`.
Chaque fonction reçoit un dictionnaire `row` et le modifie avant le chargement.

In [9]:
# Transformation 1 : Extraire les composants de la date
# La date est au format YYYY-MM-DD — on extrait chaque partie pour dim_date
def split_date(row):
    date_str = str(row.get('release_date', ''))
    parts = date_str.split('-')
    if len(parts) == 3:
        row['year']    = int(parts[0])
        row['month']   = int(parts[1])
        row['day']     = int(parts[2])
    else:
        row['year']    = 0
        row['month']   = 0
        row['day']     = 0
    return row

print('Fonction split_date définie')

Fonction split_date définie


In [10]:
# Transformation 2 : Convertir les valeurs numériques
# Les colonnes lues depuis le CSV sont des chaînes — on les convertit en nombres
def convert_numeric(row):
    row['budget']     = int(float(row.get('budget', 0) or 0))
    row['revenue']    = int(float(row.get('revenue', 0) or 0))
    row['profit']     = int(float(row.get('profit', 0) or 0))
    row['roi']        = float(row.get('roi', 0) or 0)
    row['runtime']    = int(float(row.get('runtime', 0) or 0))
    row['user_score'] = float(row.get('user_score', 0) or 0)
    row['vote_count'] = int(float(row.get('vote_count', 0) or 0))
    row['popularity'] = float(row.get('popularity', 0) or 0)
    # Metascore peut être absent si le film n'a pas été scrappé
    meta = row.get('metascore', None)
    row['metascore']  = int(float(meta)) if meta and str(meta).strip() not in ('', 'nan', 'None') else None
    return row

print('Fonction convert_numeric définie')

Fonction convert_numeric définie


In [11]:
# Transformation 3 : Nettoyer les champs texte
# Supprimer les espaces superflus et gérer les valeurs manquantes
def clean_text(row):
    champs_texte = ['title', 'original_title', 'director', 'primary_genre',
                    'studio_name', 'studio_country', 'original_language',
                    'primary_language', 'release_season', 'revenue_tier',
                    'top_cast', 'overview']
    for champ in champs_texte:
        valeur = row.get(champ, '')
        if valeur is None or str(valeur).strip() in ('', 'nan', 'None'):
            row[champ] = 'Unknown'
        else:
            row[champ] = str(valeur).strip()
    return row

print('Fonction clean_text définie')

Fonction clean_text définie


## 5. Spécification des tables de dimensions et de faits

On crée un objet `Dimension` pour chaque table de dimension et un objet `FactTable` pour chaque table de faits.
C'est exactement la même approche que dans le cours avec `Magasin_dimension`, `Produit_dimension`, etc.

In [20]:
# Dimension film
dim_movie = Dimension(
    name       = 'dim_movie',
    key        = 'movie_id',
    attributes = ['tmdb_id', 'title', 'original_title', 'director',
                  'primary_genre', 'all_genres', 'language',
                  'runtime', 'top_cast', 'overview']
)
print('dim_movie définie')

dim_movie définie


In [13]:
# Dimension date
dim_date = Dimension(
    name       = 'dim_date',
    key        = 'date_id',
    attributes = ['full_date', 'year', 'month', 'day',
                  'quarter', 'week_number', 'season']
)

print('dim_date définie')

dim_date définie


In [21]:
dim_studio = Dimension(
    name       = 'dim_studio',
    key        = 'studio_id',
    attributes = ['studio_name', 'country']
)
print('dim_studio définie')

dim_studio définie


In [22]:
dim_platform = Dimension(
    name       = 'dim_platform',
    key        = 'platform_id',
    attributes = ['platform_name', 'type', 'url']
)
print('dim_platform définie')

dim_platform définie


In [16]:
# Table de faits financiers
fact_box_office = FactTable(
    name     = 'fact_box_office',
    keyrefs  = ['movie_id', 'date_id', 'studio_id'],
    measures = ['budget', 'revenue', 'profit', 'roi', 'revenue_tier']
)

print('fact_box_office définie')

fact_box_office définie


In [17]:
# Table de faits notation
fact_ratings = FactTable(
    name     = 'fact_ratings',
    keyrefs  = ['movie_id', 'date_id', 'platform_id'],
    measures = ['user_score', 'vote_count', 'metascore', 'popularity']
)

print('fact_ratings définie')

fact_ratings définie


## 6. Pré-chargement de dim_platform

Les plateformes sont fixes et connues à l'avance (TMDB et Metacritic).
On les insère manuellement avant la boucle ETL principale.

In [23]:
dim_platform.insert({'platform_name': 'TMDB',
                     'type': 'user',
                     'url' : 'https://www.themoviedb.org'})

dim_platform.insert({'platform_name': 'Metacritic',
                     'type': 'critic',
                     'url' : 'https://www.metacritic.com'})

pgconn.commit()
print('Plateformes insérées dans dim_platform')

Plateformes insérées dans dim_platform


## 7. Boucle ETL principale — Extraction, Transformation, Chargement

On itère sur chaque ligne du fichier CSV source (`for row in movies_source`).
Pour chaque ligne :
1. On applique les fonctions de **transformation**
2. On insère dans les **dimensions** avec `insert(row)` — qui retourne la clé générée
3. On injecte les clés dans `row`
4. On insère dans les **tables de faits** avec `fact_table.insert(row)`

In [41]:
import csv

# Use Python's built-in DictReader directly instead of CSVSource
fichierSource = open('merged_movies.csv', 'r', encoding='utf-8')
movies_source = csv.DictReader(fichierSource, delimiter=',')

# Verify it has data
test = open('merged_movies.csv', 'r', encoding='utf-8')
reader = csv.reader(test)
rows_count = sum(1 for row in reader) - 1
test.close()
print(f'Fichier source contient {rows_count} lignes')

Fichier source contient 3181 lignes


In [44]:
# Reset broken transaction
pgconn.rollback()

# Reopen the file
import csv
fichierSource = open('merged_movies.csv', 'r', encoding='utf-8')
movies_source = csv.DictReader(fichierSource, delimiter=',')

# Get just the first row and test each insert one by one
row = next(iter(movies_source))
print('Ligne brute :')
print(row)
print()

# Step 1 - transformations
row = split_date(row)
row = convert_numeric(row)
row = clean_text(row)
print('Après transformations OK')

# Step 2 - dim_date
try:
    row['full_date']   = row.get('release_date', '')
    row['week_number'] = int(row.get('release_week', 0) or 0)
    row['quarter']     = int(row.get('release_quarter', 0) or 0)
    row['season']      = row.get('release_season', 'Unknown')
    row['date_id']     = dim_date.insert(row)
    print(f'dim_date OK — date_id={row["date_id"]}')
except Exception as e:
    print(f'ECHEC dim_date : {e}')
    pgconn.rollback()

# Step 3 - dim_movie
try:
    row['language'] = row.get('original_language', 'Unknown')
    row['movie_id'] = dim_movie.insert(row)
    print(f'dim_movie OK — movie_id={row["movie_id"]}')
except Exception as e:
    print(f'ECHEC dim_movie : {e}')
    pgconn.rollback()

# Step 4 - dim_studio
try:
    row['country']   = row.get('studio_country', 'Unknown')
    row['studio_id'] = dim_studio.insert(row)
    print(f'dim_studio OK — studio_id={row["studio_id"]}')
except Exception as e:
    print(f'ECHEC dim_studio : {e}')
    pgconn.rollback()

# Step 5 - fact_box_office
try:
    fact_box_office.insert(row)
    print('fact_box_office OK')
except Exception as e:
    print(f'ECHEC fact_box_office : {e}')
    pgconn.rollback()

# Step 6 - fact_ratings
try:
    row['platform_id'] = 1
    fact_ratings.insert(row)
    print('fact_ratings OK')
except Exception as e:
    print(f'ECHEC fact_ratings : {e}')
    pgconn.rollback()

pgconn.commit()
print('\nTest terminé')

Ligne brute :
{'tmdb_id': '10764', 'title': 'Quantum of Solace', 'original_title': 'Quantum of Solace', 'director': 'Marc Forster', 'primary_genre': 'Adventure', 'all_genres': 'Adventure, Action, Thriller, Crime', 'original_language': 'EN', 'primary_language': 'English', 'runtime': '106.0', 'top_cast': 'Daniel Craig, Olga Kurylenko, Mathieu Amalric', 'overview': 'Quantum of Solace continues the adventures of James Bond after Casino Royale. Betrayed by Vesper, the woman he loved, 007 fights the urge to make his latest mission personal. Pursuing his determination to uncover the truth, Bond and M interrogate Mr. White, who reveals that the organization that blackmailed Vesper is far more complex and dangerous than anyone had imagined.', 'studio_name': 'Eon Productions', 'studio_country': 'Unknown', 'primary_country': 'United Kingdom', 'release_date': '2008-10-30', 'release_year': '2008', 'release_month': '10', 'release_quarter': '4', 'release_day': '30', 'release_week': '44', 'release_sea

In [46]:
# Close everything and start completely fresh
try:
    pgconn.rollback()
    pgconn.close()
except:
    pass

import psycopg2
import pygrametl
from pygrametl.tables import Dimension, FactTable
import csv

# Fresh connection
pgconn = psycopg2.connect(
    database = 'movie_DWH',
    host     = 'localhost',
    user     = 'postgres',
    password = 'system',
    port     = 5432
)
connection = pygrametl.ConnectionWrapper(pgconn)
connection.setasdefault()
connection.execute('set search_path to "Sch_DWH"')
print('Nouvelle connexion établie')

Nouvelle connexion établie


In [47]:
dim_movie = Dimension(
    name       = 'dim_movie',
    key        = 'movie_id',
    attributes = ['tmdb_id', 'title', 'original_title', 'director',
                  'primary_genre', 'all_genres', 'language',
                  'runtime', 'top_cast', 'overview']
)
dim_date = Dimension(
    name       = 'dim_date',
    key        = 'date_id',
    attributes = ['full_date', 'year', 'month', 'day',
                  'quarter', 'week_number', 'season']
)
dim_studio = Dimension(
    name       = 'dim_studio',
    key        = 'studio_id',
    attributes = ['studio_name', 'country']
)
dim_platform = Dimension(
    name       = 'dim_platform',
    key        = 'platform_id',
    attributes = ['platform_name', 'type', 'url']
)
fact_box_office = FactTable(
    name     = 'fact_box_office',
    keyrefs  = ['movie_id', 'date_id', 'studio_id'],
    measures = ['budget', 'revenue', 'profit', 'roi', 'revenue_tier']
)
fact_ratings = FactTable(
    name     = 'fact_ratings',
    keyrefs  = ['movie_id', 'date_id', 'platform_id'],
    measures = ['user_score', 'vote_count', 'metascore', 'popularity']
)
print('Dimensions et faits définis')

Dimensions et faits définis


In [48]:
dim_platform.insert({'platform_name': 'TMDB',
                     'type': 'user',
                     'url' : 'https://www.themoviedb.org'})
dim_platform.insert({'platform_name': 'Metacritic',
                     'type': 'critic',
                     'url' : 'https://www.metacritic.com'})
pgconn.commit()
print('Plateformes insérées')

Plateformes insérées


In [49]:
fichierSource = open('merged_movies.csv', 'r', encoding='utf-8')
movies_source = csv.DictReader(fichierSource, delimiter=',')

compteur = 0
erreurs  = 0

for row in movies_source:
    try:
        row = split_date(row)
        row = convert_numeric(row)
        row = clean_text(row)

        row['full_date']   = row.get('release_date', '')
        row['week_number'] = int(row.get('release_week', 0) or 0)
        row['quarter']     = int(row.get('release_quarter', 0) or 0)
        row['season']      = row.get('release_season', 'Unknown')
        row['date_id']     = dim_date.insert(row)

        row['language'] = row.get('original_language', 'Unknown')
        row['movie_id'] = dim_movie.insert(row)

        row['country']   = row.get('studio_country', 'Unknown')
        row['studio_id'] = dim_studio.insert(row)

        fact_box_office.insert(row)

        row['platform_id'] = 1
        fact_ratings.insert(row)

        compteur += 1

    except Exception as e:
        erreurs += 1
        pgconn.rollback()
        if erreurs <= 3:
            print(f'Erreur ligne {compteur + erreurs} : {e}')
        continue

print(f'ETL terminé — {compteur} lignes chargées, {erreurs} erreurs')

ETL terminé — 3181 lignes chargées, 0 erreurs


In [50]:
pgconn.commit()
pgconn.close()
print('Commit effectué et connexion fermée')

Commit effectué et connexion fermée


## 8. Validation et fermeture de la connexion

`commit()` rend toutes les insertions persistantes dans PostgreSQL.
Sans `commit()`, aucune donnée n'est réellement sauvegardée.
`close()` libère la connexion.

In [25]:
# Valider toutes les insertions
pgconn.commit()
print('Données validées (commit effectué)')

# Fermer la connexion
pgconn.close()
print('Connexion fermée')

Données validées (commit effectué)
Connexion fermée


## 9. Vérification du chargement

On rouvre une connexion pour vérifier que les données ont bien été chargées dans PostgreSQL.
On utilise `cursor()`, `execute()` et `fetchall()` exactement comme dans le cours.

In [59]:
pgconn_verif = psycopg2.connect(
    database = 'movie_DWH',
    host     = 'localhost',
    user     = 'postgres',
    password = 'system',
    port     = 5432
)
cur = pgconn_verif.cursor()
cur.execute('SET search_path TO "Sch_DWH"')

# Compter les lignes dans chaque table
tables = ['dim_movie', 'dim_date', 'dim_studio', 'dim_platform',
          'fact_box_office', 'fact_ratings']
for table in tables:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    print(f'{table} : {cur.fetchone()[0]} lignes')

dim_movie : 3181 lignes
dim_date : 3181 lignes
dim_studio : 3181 lignes
dim_platform : 4 lignes
fact_box_office : 3181 lignes
fact_ratings : 3181 lignes


In [61]:
# Afficher les 5 premiers films chargés dans dim_movie
cur.execute('SELECT * FROM dim_movie LIMIT 5')
rows = cur.fetchall()
for row in rows:
    print(row)

(1, 10764, 'Quantum of Solace', 'Quantum of Solace', 'Marc Forster', 'Adventure', 'Adventure, Action, Thriller, Crime', 'EN', 106, 'Daniel Craig, Olga Kurylenko, Mathieu Amalric', 'Quantum of Solace continues the adventures of James Bond after Casino Royale. Betrayed by Vesper, the woman he loved, 007 fights the urge to make his latest mission personal. Pursuing his determination to uncover the truth, Bond and M interrogate Mr. White, who reveals that the organization that blackmailed Vesper is far more complex and dangerous than anyone had imagined.')
(2, 20662, 'Robin Hood', 'Robin Hood', 'Ridley Scott', 'Action', 'Action, Adventure', 'EN', 140, 'Russell Crowe, Cate Blanchett, Max von Sydow', "When soldier Robin happens upon the dying Robert of Loxley, he promises to return the man's sword to his family in Nottingham. There, he assumes Robert's identity; romances his widow, Marion; and draws the ire of the town's sheriff and King John's henchman, Godfrey.")
(3, 2268, 'The Golden Comp

In [62]:
# Afficher les 5 premières lignes de fact_box_office
cur.execute('SELECT * FROM fact_box_office LIMIT 5')
rows = cur.fetchall()
for row in rows:
    print(row)

(1, 1, 1, 1, 200000000, 586090727, 386090727, 1.9305, 'Blockbuster (500M+)')
(2, 2, 2, 2, 200000000, 310669540, 110669540, 0.5533, 'Hit (100M-500M)')
(3, 3, 3, 3, 180000000, 372234864, 192234864, 1.068, 'Hit (100M-500M)')
(4, 4, 4, 4, 200000000, 783766341, 583766341, 2.9188, 'Blockbuster (500M+)')
(5, 5, 5, 5, 200000000, 743559607, 543559607, 2.7178, 'Blockbuster (500M+)')


In [63]:
# Requête de jointure : top 5 films par revenus
# Utilisation de INNER JOIN comme vu dans le cours
cur.execute("""
    SELECT m.title, m.director, m.primary_genre,
           f.budget, f.revenue, f.profit, f.roi
    FROM fact_box_office f
    INNER JOIN dim_movie m ON f.movie_id = m.movie_id
    ORDER BY f.revenue DESC
    LIMIT 5
""")
rows = cur.fetchall()
print('Top 5 films par revenus :')
for row in rows:
    print(row)

Top 5 films par revenus :
('Shrek 2', 'Andrew Adamson', 'Adventure', 150000000, 919838758, 769838758, 5.1323)
('Harry Potter and the Goblet of Fire', 'Mike Newell', 'Adventure', 150000000, 895921036, 745921036, 4.9728)
('Ice Age: Dawn of the Dinosaurs', 'Carlos Saldanha', 'Animation', 90000000, 886686817, 796686817, 8.8521)
('Ice Age: Continental Drift', 'Steve Martino', 'Animation', 95000000, 877244782, 782244782, 8.2342)
('Harry Potter and the Chamber of Secrets', 'Chris Columbus', 'Adventure', 100000000, 876688482, 776688482, 7.7669)


In [64]:
# Requête GROUP BY : revenus totaux par genre
cur.execute("""
    SELECT m.primary_genre, COUNT(*) as nb_films,
           SUM(f.revenue) as revenu_total
    FROM fact_box_office f
    INNER JOIN dim_movie m ON f.movie_id = m.movie_id
    GROUP BY m.primary_genre
    ORDER BY revenu_total DESC
    LIMIT 10
""")
rows = cur.fetchall()
print('Revenus par genre :')
for row in rows:
    print(row)

Revenus par genre :
('Action', 571, Decimal('73842919609'))
('Adventure', 271, Decimal('56108800948'))
('Comedy', 634, Decimal('53037118804'))
('Drama', 745, Decimal('51252164870'))
('Animation', 94, Decimal('24751140220'))
('Fantasy', 92, Decimal('16166326541'))
('Science Fiction', 77, Decimal('13565346696'))
('Horror', 197, Decimal('13252095127'))
('Thriller', 117, Decimal('11384020332'))
('Crime', 141, Decimal('9408596694'))


In [ ]:
# Fermer le curseur et la connexion de vérification
cur.close()
pgconn_verif.close()
print('Vérification terminée — ETL complet et validé')

Vérification terminée — ETL complet et validé


: 